In [23]:
import math
class obj:
    def __init__(self,value=0.0,gradient=0.0,children1=None,children2=None):
        self.value=value;
        self.gradient=gradient;
        self.children1=children1;
        self.children2=children2;
        self._backward=lambda:None
    def __add__(self1, other):
        out=obj(self1.value+other.value)
        
        def _backward():
            self1.gradient+=1
            other.gradient+=1
            out.children1=self1
            out.children2=other
        out._backward=_backward
        
        return out
    def __sub__(self1, other):
        out=obj(self1.value-other.value)
        
        def _backward():
            self1.gradient+=1
            other.gradient+=1
            out.children1=self1
            out.children2=other
        out._backward=_backward
        
        return out    
    def __mul__(self1, other):
        out=obj(self1.value*other.value)
        
        def _backward():
            self1.gradient+=other.value;
            other.gradient+=self1.value; 
            out.children1=self1
            out.children2=other  
        out._backward=_backward
        return out
    def tanh(self):
        x=self.value
        o=(math.exp(x)-math.exp(-x))/(math.exp(x)+math.exp(-x))
        
        def _backward():
            self.gradient+=(1-(o**2))
        out=obj(o)
        out._backward=_backward
        out.children1=self
        return out
    
def backprop(obj):
   
    obj._backward();
    if(obj.children1 is not None):
        backprop(obj.children1)
        
    if(obj.children2 is not None):
         backprop(obj.children2)


In [53]:
import random

class neuron:
    def __init__(self,inp):
      
        self.weight=[]
        self.bias=obj(random.randint(0,3))
        for i in range(0,inp):
            self.weight.append(obj(random.randint(1,100)/100))
            
    def compute(self,inputvalue):
        out=obj(0)
        for i in range(0,len(self.weight)):
          out += self.weight[i]*inputvalue[i]
        out +=self.bias
        outfinal=out.tanh()
       
        return outfinal;
    
class layer:
    def __init__(self,number,inp):
        self.list=[]
        for i in range(0,number):
            self.list.append(neuron(inp))   
            
    def compute(self,inputvalue):
        if(len(inputvalue)!=len(self.list)):
             print("Error len of input does not man no.of neuron in layer")
        out=[]
        for i in range(0,len(self.list)):
             
             out.append(self.list[i].compute(inputvalue))
            
        return out
      
class transformer:
    def __init__(self,numberarr,inpArr):
        self.list=[]
        for i in range(0,len(numberarr)):
            self.list.append(layer(numberarr[i],inpArr))
            
    def compute(self,inputlayer):
        layer=inputlayer
        for i in range(0,len(self.list)):
            layer= self.list[i].compute(layer)
        return layer
    
def loss(output,target):
    if(len(output)!=len(target)):
        print("error in loss fucntion incorrect lengths")
    loss=obj(0.0)
    for i in range(0,len(output)):
       d= (target[i]-output[i])
       loss+=d**obj(2)
    loss=(loss/(obj(len(output))))
    return loss




    



In [54]:
i=[3,3,3]
tr=transformer(i,3)
inpt=[obj(2),obj(8),obj(1)]
output=tr.compute(inpt)
expected=[obj(0),obj(0),obj(0)]
print(*(obj.value for obj in output))
print(loss(output,expected).value)

0.9997024247142131 0.9674589102619967 0.9701484224002755


TypeError: unsupported operand type(s) for ** or pow(): 'obj' and 'obj'